## 1. 커스텀 Tool 설계 원칙

ReAct 에이전트에서 Tool은 에이전트가 외부 세계와 상호작용하는 유일한 통로이다. 따라서 Tool의 설계 품질이 에이전트의 전체 성능을 좌우한다. 다음은 커스텀 Tool을 설계할 때 반드시 지켜야 할 핵심 원칙들이다.

### 1.1 단일 책임 원칙 (Single Responsibility)

하나의 Tool은 **하나의 명확한 기능**만 수행해야 한다. 뉴스 검색과 요약을 하나의 Tool에 넣는 것이 아니라, 검색 Tool과 요약 Tool을 분리하여 구현한다. 이렇게 하면 LLM이 각 도구의 역할을 명확히 이해할 수 있고, 도구 조합의 유연성이 높아진다.

### 1.2 명확한 Tool 설명문 작성

Tool의 설명문(description)은 LLM이 도구를 올바르게 선택하는 데 결정적인 역할을 한다. 설명문에는 다음 요소가 포함되어야 한다:

- **기능 설명**: 이 도구가 무엇을 하는지 한 문장으로 명시
- **입력 형식**: 어떤 형식의 입력을 기대하는지 구체적으로 명시
- **사용 예시**: 실제 호출 예시를 포함하여 LLM이 패턴을 학습하도록 유도

### 1.3 입력/출력 형식 표준화

모든 Tool은 **문자열 입력 -> 문자열 출력** 형태를 유지하는 것이 좋다. ReAct 프레임워크에서 Action과 Observation은 텍스트 기반이므로, 일관된 문자열 인터페이스를 유지하면 에이전트와의 통합이 용이하다.

### 1.4 에러 처리와 Fallback 전략

외부 API를 호출하는 Tool은 반드시 예외 처리를 포함해야 한다. 네트워크 오류, 타임아웃, API 제한 등 다양한 실패 상황에 대비하여:

- try/except로 예외를 포착하고 의미 있는 오류 메시지를 반환
- 타임아웃 설정으로 무한 대기 방지
- 재시도 로직으로 일시적 오류에 대응
- 대체 데이터 소스나 기본값을 준비

## 2. 뉴스 검색 커스텀 Tool 개발

첫 번째 커스텀 Tool로 Google News RSS 피드를 활용한 **뉴스 검색 Tool**을 구현한다. 이 Tool은 실시간 뉴스 데이터를 수집하여 에이전트에게 최신 정보를 제공하는 역할을 한다.

### RSS 기반 뉴스 검색의 장점

- 별도의 API 키 없이 무료로 사용 가능
- XML 형식으로 구조화된 데이터 제공
- 실시간 최신 뉴스 접근 가능
- 한국어 뉴스 지원